In [1]:
import pandas as pd

# Convert CSV files into dataframes
sampled_df = pd.read_csv('../data/sample/sample-pairwise.csv')
product_df = pd.read_csv('../data/intermediary/product-info.csv')

In [3]:
def compute_hybrid_substitution_score(product_df, pairwise_df, output_csv=None):
    if output_csv:
        with open(output_csv, "w") as f:
            f.write("product_id,substitute_id,score,rank\n")

    pairwise_df_i = pairwise_df.groupby("product_i")
    pairwise_df_j = pairwise_df.groupby("product_j")

    for row in product_df.itertuples(index=False):
        product_id = row.product_id

        product_probs = pairwise_df_i.get_group(product_id) if product_id in pairwise_df_i.groups else pd.DataFrame()
        product_j_probs = pairwise_df_j.get_group(product_id) if product_id in pairwise_df_j.groups else pd.DataFrame()

        # Reverse relationships
        products_rev = product_j_probs.rename(columns={
            'product_i': 'product_j',
            'product_j': 'product_i',
            'P_i': 'P_j',
            'P_j': 'P_i'
        })

        product_probs_df = pd.concat([product_probs, products_rev], ignore_index=True)
        if product_probs_df.empty:
            continue

        # Compute metrics
        product_probs_df['jaccard'] = product_probs_df.apply(lambda x: x.P_ij / (x.P_i + x.P_j - x.P_ij), axis=1)
        product_probs_df['conditional'] = product_probs_df.apply(lambda x: ((x.P_ij / x.P_i) + (x.P_ij / x.P_j)) / 2, axis=1)
        product_probs_df['substitution_index'] = product_probs_df.apply(lambda x: ((x.P_i * x.P_j) - x.P_ij) / (x.P_i * x.P_j), axis=1)

        # Normalize
        for col in ['jaccard', 'conditional', 'substitution_index']:
            product_probs_df[col] = (product_probs_df[col] - product_probs_df[col].min()) / (product_probs_df[col].max() - product_probs_df[col].min())

        # Weighted hybrid score
        product_probs_df['score'] = (
            0.5 * product_probs_df['substitution_index'] +
            0.3 * product_probs_df['jaccard'] +
            0.2 * product_probs_df['conditional']
        )

        # Rank
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False)
        product_probs_df = product_probs_df.dropna(subset=['score'])
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False).astype(int)
        
        # Keep only top 20
        top_substitutes_df = product_probs_df[product_probs_df['rank'] <= 20].copy()

        # Select and rename columns before saving
        top_substitutes_df = top_substitutes_df[['product_i', 'product_j', 'score', 'rank']]
        top_substitutes_df.columns = ['product_id', 'substitute_id', 'score', 'rank']

        # Append to CSV
        if output_csv:
            top_substitutes_df.to_csv(output_csv, mode='a', index=False, header=False)
        else:
            return top_substitutes_df  # For testing

    print(f"Completed substitute calculations. Saved to {output_csv if output_csv else 'DataFrame'}")

compute_hybrid_substitution_score(product_df, sampled_df, output_csv="../data/sample/obj1/sample-substitutes.csv")


Completed substitute calculations. Saved to ../data/sample/obj1/sample-substitutes.csv


In [ ]:
# Category validation

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def validate_substitutes(subs_df, product_info_df):
    """
    Validate a substitution dataframe by checking if product and substitute belong 
    to the same department and/or aisle.

    Parameters:
    - subs_df: DataFrame with ['product_id', 'substitute_id', 'score', 'rank']
    - product_info_df: DataFrame with ['product_id', 'department', 'aisle']

    Returns:
    - results: dict containing separate and overall validation metrics
    - validated_df: DataFrame with added columns for validation results
    """

    # Map product_id to department and aisle
    dept_map = product_info_df.set_index('product_id')['department'].to_dict()
    aisle_map = product_info_df.set_index('product_id')['aisle'].to_dict()

    # Only evaluate the top 5 substitutes
    subs_df = subs_df[subs_df['rank'] <= 5].copy()
    # Add department and aisle info to subs_df
    subs_df['product_dept'] = subs_df['product_id'].map(dept_map)
    subs_df['substitute_dept'] = subs_df['substitute_id'].map(dept_map)
    subs_df['product_aisle'] = subs_df['product_id'].map(aisle_map)
    subs_df['substitute_aisle'] = subs_df['substitute_id'].map(aisle_map)

    # Check matches
    subs_df['same_department'] = subs_df['product_dept'] == subs_df['substitute_dept']
    subs_df['same_aisle'] = subs_df['product_aisle'] == subs_df['substitute_aisle']
    subs_df['valid_substitution'] = subs_df['same_department'] & subs_df['same_aisle']

    # Compute base true and predicted labels
    y_true = [1] * len(subs_df)  # expecting all to be valid (theoretical ideal)
    dept_pred = subs_df['same_department'].astype(int)
    aisle_pred = subs_df['same_aisle'].astype(int)
    overall_pred = subs_df['valid_substitution'].astype(int)

    # Helper function for metrics
    def compute_metrics(y_pred):
        return {
            'accuracy': y_pred.mean(),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'valid_pairs': y_pred.sum(),
            'total_pairs': len(y_pred)
        }

    # Compute metrics by level
    results = {
        'department_metrics': compute_metrics(dept_pred),
        'aisle_metrics': compute_metrics(aisle_pred),
        'combined_metrics': compute_metrics(overall_pred),
    }

    # Add extra breakdown
    breakdown = subs_df.groupby(['same_department', 'same_aisle']).size().reset_index(name='count')
    results['breakdown'] = breakdown

    return results, subs_df


from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

def validate_substitutes_by_department(subs_df, product_info_df, top_n=5):
    """
    Validate a substitution dataframe by checking if product and substitute belong 
    to the same department and/or aisle, computing metrics per department, 
    and summarizing across departments.

    Parameters:
    - subs_df: DataFrame with ['product_id', 'substitute_id', 'score', 'rank']
    - product_info_df: DataFrame with ['product_id', 'department', 'aisle']
    - top_n: Only evaluate top N substitutes per product

    Returns:
    - dept_metrics: dict mapping department -> metrics
    - summary_metrics: DataFrame with mean, std, min, max per metric across departments
    - validated_df: subs_df with added validation columns
    """

    # Map product_id to department and aisle
    dept_map = product_info_df.set_index('product_id')['department'].to_dict()
    aisle_map = product_info_df.set_index('product_id')['aisle'].to_dict()

    # Filter to top N substitutes
    subs_df = subs_df[subs_df['rank'] <= top_n].copy()

    # Add department and aisle info
    subs_df['product_dept'] = subs_df['product_id'].map(dept_map)
    subs_df['substitute_dept'] = subs_df['substitute_id'].map(dept_map)
    subs_df['product_aisle'] = subs_df['product_id'].map(aisle_map)
    subs_df['substitute_aisle'] = subs_df['substitute_id'].map(aisle_map)

    # Check matches
    subs_df['same_department'] = subs_df['product_dept'] == subs_df['substitute_dept']
    subs_df['same_aisle'] = subs_df['product_aisle'] == subs_df['substitute_aisle']
    subs_df['valid_substitution'] = subs_df['same_department'] & subs_df['same_aisle']

    # Helper function to compute metrics
    def compute_metrics(y_pred):
        y_true = [1] * len(y_pred)  # ideal: all should be valid
        return {
            'accuracy': y_pred.mean(),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'valid_pairs': y_pred.sum(),
            'total_pairs': len(y_pred)
        }

    # Compute metrics per department
    dept_metrics = {}
    all_metrics_records = []
    for dept, group in subs_df.groupby('product_dept'):
        dept_pred = group['same_department'].astype(int)
        aisle_pred = group['same_aisle'].astype(int)
        combined_pred = group['valid_substitution'].astype(int)

        metrics = {
            'department_metrics': compute_metrics(dept_pred),
            'aisle_metrics': compute_metrics(aisle_pred),
            'combined_metrics': compute_metrics(combined_pred)
        }
        dept_metrics[dept] = metrics

        # Flatten metrics for summary
        for level, vals in metrics.items():
            record = {'department': dept, 'level': level}
            record.update(vals)
            all_metrics_records.append(record)

    # Summary across departments
    summary_metrics = pd.DataFrame(all_metrics_records)

    # Select only numeric columns for aggregation
    numeric_cols = summary_metrics.select_dtypes(include=np.number).columns

    # Aggregate numeric columns per level
    summary_metrics = summary_metrics.groupby('level')[numeric_cols].agg(['mean', 'std', 'min', 'max'])

    return dept_metrics, summary_metrics, subs_df



substitutes_df = pd.read_csv("../data/sample/obj1/sample-substitutes.csv")

results, validated_df = validate_substitutes(substitutes_df, product_df)

validated_df.to_csv("../data/sample/obj1/sample-results.csv", index=False)

print("Department-level metrics:")
print(results['department_metrics'])

print("\nAisle-level metrics:")
print(results['aisle_metrics'])

print("\nCombined (Dept + Aisle) metrics:")
print(results['combined_metrics'])

print("\nBreakdown:")
print(results['breakdown'])


dept_metrics, summary_metrics, dept_validated_df = validate_substitutes_by_department(substitutes_df, product_df, top_n=5)

dept_validated_df.to_csv("../data/sample/obj1/sample-dept-results.csv", index=False)

print("=== Per-Department Metrics ===")
for dept, metrics in dept_metrics.items():
    print(f"\nDepartment: {dept}")
    print("  Department-level metrics:", metrics['department_metrics'])
    print("  Aisle-level metrics:", metrics['aisle_metrics'])
    print("  Combined (Dept + Aisle) metrics:", metrics['combined_metrics'])

print("\n=== Summary Metrics Across Departments ===")
print(summary_metrics)

print("\n=== Sample of Validated Substitutes ===")
print(validated_df.head())



Department-level metrics:
{'accuracy': 0.18044791868382884, 'precision': 1.0, 'recall': 0.18044791868382884, 'f1_score': 0.3057278780838107, 'valid_pairs': 45660, 'total_pairs': 253037}

Aisle-level metrics:
{'accuracy': 0.09790663025565431, 'precision': 1.0, 'recall': 0.09790663025565431, 'f1_score': 0.17835146916428796, 'valid_pairs': 24774, 'total_pairs': 253037}

Combined (Dept + Aisle) metrics:
{'accuracy': 0.09790663025565431, 'precision': 1.0, 'recall': 0.09790663025565431, 'f1_score': 0.17835146916428796, 'valid_pairs': 24774, 'total_pairs': 253037}

Breakdown:
   same_department  same_aisle   count
0            False       False  207377
1             True       False   20886
2             True        True   24774
=== Per-Department Metrics ===

Department: alcohol
  Department-level metrics: {'accuracy': 0.2768698327244761, 'precision': 1.0, 'recall': 0.2768698327244761, 'f1_score': 0.43366962806806203, 'valid_pairs': 1440, 'total_pairs': 5201}
  Aisle-level metrics: {'accurac